# Thu thập dữ liệu có cấu trúc (`ingest_structure_data`)

_Bản sao/chỉnh từ luồng giá; output ghi vào `data-lake/raw/Structure_Data/...`._

**Đồ án:** Hệ thống Data Pipeline và tra cứu phân tích thị trường chứng khoán Việt Nam đa nguồn.

**Mục đích:** Tải lịch sử giá OHLCV có cấu trúc qua thư viện **vnstock** , chuẩn hóa/làm sạch, ghi vào **Data Lake** dạng Parquet theo partition Hive-style `date=YYYY-MM-DD` (ngày chạy pipeline).

**Nguồn `Quote`:** ưu tiên **KBS** (`PRIMARY_QUOTE_SOURCE`), fallback **VCI** (`FALLBACK_QUOTE_SOURCE`).

**Đầu ra:** `../data-lake/raw/price/date=<YYYY-MM-DD>/<Mã>.parquet` (một file mỗi mã); chỉ số: `../data-lake/raw/index/date=<YYYY-MM-DD>/<MÃ_CHỈ_SỐ>.parquet` (theo `INDEX_TICKERS`, cùng API `Quote.history` — xem `.cursor/rules/instructions.md`).

**Cách chạy:** Chọn kernel Python của project (venv), chạy lần lượt các ô: import → helper → định nghĩa hàm → `main()` (cổ phiếu) → ô **index** → các ô còn lại → QC cuối.


In [1]:
# vnstock Quote: PRIMARY = KBS, FALLBACK = VCI (nguồn ổn định).
# TCBS đã deprecated theo tài liệu vnstock — không dùng làm source.
import os

# Đặt UTF-8 để subprocess/thread đọc output không bị cp1252 trên Windows.
os.environ.setdefault("PYTHONUTF8", "1")
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("LANG", "C.UTF-8")
os.environ.setdefault("LC_ALL", "C.UTF-8")
# Windows-specific encoding fix
os.environ["PYTHONLEGACYWINDOWSSTDIO"] = "0"

import logging
import time
from datetime import date, datetime, timezone
from pathlib import Path

import pandas as pd
from vnstock import Listing, Company, Finance, Quote

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# ========== RATE LIMIT CONFIG ==========
RATE_LIMIT_RPM = 10  # 10 requests/phút
DELAY_BETWEEN_CATEGORIES_SEC = 60  # Pause 60s giữa các category
MAX_TICKERS_PER_RUN = 20  # Giới hạn 20 mã/lần chạy

TICKERS: list[str] = [
    # VN30 hiện tại (20 mã)
    "ACB","BCM","BID","BVH","CTG","FPT","GAS","GVR","HDB","HPG",
    "MBB","MSN","MWG","PLX","POW","SAB","SHB","SSI","STB","TCB",
    # # Thêm 30 mã đa ngành
    # "VCB","VHM","VIC","VJC","VNM","VPB","VRE","TPB","VIB","HVN",
    # "REE","PNJ","GMD","DGC","DPM","DCM","HSG","KDC","QNS","SCS",
    
    # "NVL","PDR","DXG","KDH","HDG","EVF","KBC","AGG","TCH","VPI",
]
PRIMARY_QUOTE_SOURCE = "kbs"
FALLBACK_QUOTE_SOURCE = "vci"

# Chỉ số thị trường VN — vnstock Quote.history (1D), mã symbol theo tài liệu vnstock (KBS/VCI).
INDEX_TICKERS: list[str] = [
    "VNINDEX",
    "VN30",
    "HNXINDEX",
    "HNX30",
    "UPCOMINDEX",
]

assert len(TICKERS) == 20, f"Expected 20 tickers, got {len(TICKERS)}"
assert len(set(TICKERS)) == len(TICKERS), "Duplicate tickers found"

In [2]:
import os
import sys
import warnings
import threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Suppress encoding errors từ subprocess threads
original_hook = threading.excepthook
def custom_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass  # Ignore encoding errors from threads
    else:
        original_hook(args)
threading.excepthook = custom_hook

# Suppress non-critical warnings
warnings.filterwarnings("ignore")

print("[OK] UTF-8 guard enabled for current kernel session")

[OK] UTF-8 guard enabled for current kernel session


In [3]:
def save_raw(df: pd.DataFrame, category: str, run_date: str, name: str) -> str:
    """Lưu raw dạng Parquet (encoding utf-8-sig).

    Path:
      ../data-lake/raw/{category}/date={run_date}/{name}.parquet
    """
    base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data-lake", "raw", "Structure_Data", category))
    partition = os.path.join(base_dir, f"date={run_date}")
    os.makedirs(partition, exist_ok=True)

    # Đối với ticker, luôn chuẩn hóa UPPER để đồng nhất tên file.
    if category in ("price", "index", "financial_ratio"):
        safe_name = str(name).upper().strip()
    else:
        safe_name = str(name).strip()

    parquet_path = os.path.join(partition, f"{safe_name}.parquet")
    csv_path = os.path.join(partition, f"{safe_name}.csv")

    try:
        df.to_parquet(parquet_path, engine="pyarrow", index=False)
    except ImportError:
        df.to_parquet(parquet_path, engine="fastparquet", index=False)

    # df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    # logger.info("Đã ghi CSV (%s dòng) → %s", len(df), csv_path)

    logger.info("Đã ghi %s dòng → %s", len(df), parquet_path)
    return parquet_path


In [4]:
import re
import os
from pathlib import Path
from datetime import timedelta

# Rate limit state
_last_request_time = 0.0


def _ensure_ingestion_workdir():
    """Thiết lập working directory (notebook ingest_structure_data hoặc ingest_price_data)."""
    cwd = Path.cwd()
    if (cwd / "ingest_structure_data.ipynb").is_file() or (
        cwd / "ingest_price_data.ipynb"
    ).is_file():
        return
    ing = cwd / "ingestion"
    if (ing / "ingest_structure_data.ipynb").is_file() or (
        ing / "ingest_price_data.ipynb"
    ).is_file():
        os.chdir(ing)


def _date_years_ago(ref_date, years):
    """Tính ngày cách đây N năm."""
    try:
        return ref_date.replace(year=ref_date.year - years)
    except ValueError:
        return ref_date.replace(year=ref_date.year - years, day=28)


def _wait_for_rate_limit():
    """Throttle requests."""
    global _last_request_time
    now = time.time()
    elapsed = now - _last_request_time
    min_interval = 60.0 / RATE_LIMIT_RPM
    if elapsed < min_interval:
        time.sleep(min_interval - elapsed)
    _last_request_time = time.time()


def _extract_wait_seconds(msg, default=60):
    """Extract wait time from error message."""
    match = re.search(r"(\d+)\s*(?:giây|s|second)", msg)
    return int(match.group(1)) if match else default


def transform_data(df):
    """Transform data: convert date, rename columns."""
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    if "tradingdate" not in df.columns and "time" in df.columns:
        df = df.rename(columns={"time": "tradingdate"})
    df = df.rename(columns={
        "tradingdate": "tradingDate", "open": "open", "high": "high",
        "low": "low", "close": "close", "volume": "volume", "value": "value"
    })
    if "tradingDate" in df.columns:
        df["tradingDate"] = pd.to_datetime(
            df["tradingDate"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")
    if "tradingdate" in df.columns:
        df["tradingdate"] = pd.to_datetime(
            df["tradingdate"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")
    return df


def _validate_ohlcv_frame(df: pd.DataFrame) -> tuple[bool, str]:
    """Kiểm tra DataFrame OHLCV sau transform_data (index / sanity check)."""
    if df is None or df.empty:
        return False, "empty"
    n = len(df)
    if n < 200:
        return False, f"rows_{n}_lt_200"
    date_col = None
    for c in ("tradingDate", "tradingdate", "time"):
        if c in df.columns:
            date_col = c
            break
    if date_col is None:
        return False, "no_date_column"
    dt = pd.to_datetime(df[date_col], errors="coerce")
    if dt.isna().all():
        return False, "dates_all_invalid"
    if "close" not in df.columns:
        return False, "no_close"
    close = pd.to_numeric(df["close"], errors="coerce")
    if close.isna().all():
        return False, "close_all_nan"
    if "high" in df.columns and "low" in df.columns:
        h = pd.to_numeric(df["high"], errors="coerce")
        l = pd.to_numeric(df["low"], errors="coerce")
        both = h.notna() & l.notna()
        if both.any():
            bad = (h < l) & both
            err_ratio = float(bad.sum() / max(int(both.sum()), 1))
            if err_ratio > 0.05:
                return False, f"high_lt_low_ratio_{err_ratio:.3f}"
    return True, "ok"


def _log_ohlcv_quality(label: str, df: pd.DataFrame, src_used: str | None) -> None:
    """Log ticker/index: src, rows, dates, % missing close/value."""
    n = len(df)
    date_col = next(
        (c for c in ("trading_date", "tradingDate", "tradingdate") if c in df.columns),
        None,
    )
    if date_col:
        dt = pd.to_datetime(df[date_col], errors="coerce")
        dmin, dmax = dt.min(), dt.max()
        dmin_s = dmin.isoformat()[:10] if pd.notna(dmin) else "NA"
        dmax_s = dmax.isoformat()[:10] if pd.notna(dmax) else "NA"
    else:
        dmin_s = dmax_s = "NA"
    pct_close = (
        float(df["close"].isna().mean() * 100)
        if "close" in df.columns and n
        else 100.0
    )
    pct_val = (
        float(df["value"].isna().mean() * 100)
        if "value" in df.columns and n
        else 100.0
    )
    src = (src_used or "none").upper()
    logger.info(
        "%s | src_used=%s rows=%s min_date=%s max_date=%s "
        "pct_missing_close=%.2f%% pct_missing_value=%.2f%%",
        label,
        src,
        n,
        dmin_s,
        dmax_s,
        pct_close,
        pct_val,
    )


def _apply_value_derivation(out: pd.DataFrame) -> pd.DataFrame:
    """Điền value = close * volume khi thiếu; đánh dấu value_is_derived."""
    out = out.copy()
    close = pd.to_numeric(out["close"], errors="coerce")
    vol = pd.to_numeric(out["volume"], errors="coerce")
    v = pd.to_numeric(out["value"], errors="coerce")
    derived_raw = close * vol
    need = v.isna()
    out["value"] = v.where(~need, derived_raw)
    out["value_is_derived"] = need.fillna(True).astype(bool)
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    return out


def build_price_target_schema(
    df: pd.DataFrame,
    ticker: str,
    run_date: str,
    *,
    src_used: str | None = None,
    instrument_type: str = "stock",
) -> pd.DataFrame:
    """Giữ cột gốc provider; thêm metadata, trading_date, value derived nếu cần."""
    out = df.copy()

    if "ticker" not in out.columns:
        out.insert(0, "ticker", ticker)
    fetched_ts = datetime.now(timezone.utc).isoformat()
    out["fetched_at"] = fetched_ts
    out["ingested_at"] = run_date
    out["source"] = (src_used or "unknown").lower()
    out["instrument_type"] = instrument_type

    for col in ["open", "high", "low", "close", "volume", "value"]:
        if col not in out.columns:
            out[col] = pd.NA

    for col in ["open", "high", "low", "close", "volume", "value"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = _apply_value_derivation(out)

    if "tradingDate" in out.columns:
        out["trading_date"] = pd.to_datetime(
            out["tradingDate"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")
    elif "tradingdate" in out.columns:
        out["trading_date"] = pd.to_datetime(
            out["tradingdate"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")

    if "is_suspicious" not in out.columns:
        hl_bad = (
            out["high"].notna()
            & out["low"].notna()
            & (out["high"] < out["low"])
        )
        out["is_suspicious"] = (out["close"] < 0) | hl_bad

    priority = [
        "ticker",
        "trading_date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "value",
        "value_is_derived",
        "source",
        "instrument_type",
        "ingested_at",
        "fetched_at",
        "is_suspicious",
    ]
    first = [c for c in priority if c in out.columns]
    rest = [c for c in out.columns if c not in first]
    out = out[first + rest]

    logger.info("%s — cột output: %s", ticker, out.columns.tolist())
    return out


def fetch_ticker_data(ticker, start, end) -> tuple[pd.DataFrame, str | None]:
    """Fetch OHLCV; trả về (df, src_used) để lineage đúng."""
    sym = ticker.upper().strip()
    last_err: str | None = None
    for src in (PRIMARY_QUOTE_SOURCE, FALLBACK_QUOTE_SOURCE):
        try:
            _wait_for_rate_limit()
            quote = Quote(source=src, symbol=sym)
            df = quote.history(start=start, end=end, interval="1D")
            if df is not None and not df.empty:
                logger.info("Lấy %s từ %s", sym, src.upper())
                return df.copy(), src
            last_err = "empty_response"
            if src == PRIMARY_QUOTE_SOURCE:
                logger.warning(
                    "fallback KBS -> VCI cho %s (KBS trả rỗng)",
                    sym,
                )
        except Exception as e:
            last_err = str(e)[:200]
            logger.warning(
                "Fetch %s (%s) lỗi: %s",
                sym,
                src,
                str(e)[:120],
            )
            if src == PRIMARY_QUOTE_SOURCE:
                logger.warning(
                    "fallback KBS -> VCI cho %s (exception từ KBS)",
                    sym,
                )
            continue
    if last_err:
        logger.warning("Không lấy được %s sau cả 2 nguồn: %s", sym, last_err)
    return pd.DataFrame(), None


def fetch_index_data(
    index_code: str, start: str, end: str
) -> tuple[pd.DataFrame, str | None]:
    """Index: fetch + validate; KBS fail validation/exception -> VCI; skip nếu cả 2 fail."""
    code = index_code.upper().strip()
    last_reason: str | None = None
    kbs_fail_reason: str | None = None

    for src in (PRIMARY_QUOTE_SOURCE, FALLBACK_QUOTE_SOURCE):
        try:
            _wait_for_rate_limit()
            quote = Quote(source=src, symbol=code)
            raw = quote.history(start=start, end=end, interval="1D")
            if raw is None or raw.empty:
                last_reason = "empty"
                if src == PRIMARY_QUOTE_SOURCE:
                    kbs_fail_reason = "empty"
                    logger.warning(
                        "fallback KBS -> VCI cho index %s (KBS trả rỗng)",
                        code,
                    )
                continue
            cleaned = transform_data(raw.copy())
            ok, reason = _validate_ohlcv_frame(cleaned)
            if not ok:
                last_reason = reason
                if src == PRIMARY_QUOTE_SOURCE:
                    kbs_fail_reason = reason
                    logger.warning(
                        "fallback KBS -> VCI cho index %s (validation KBS fail: %s)",
                        code,
                        reason,
                    )
                else:
                    logger.warning(
                        "index %s: VCI validation fail (%s)",
                        code,
                        reason,
                    )
                continue
            logger.info("Lấy %s từ %s (validation OK)", code, src.upper())
            if src == FALLBACK_QUOTE_SOURCE and kbs_fail_reason:
                logger.info(
                    "index %s dùng VCI sau KBS không đạt (lý do: %s)",
                    code,
                    kbs_fail_reason,
                )
            return cleaned, src
        except Exception as e:
            last_reason = str(e)[:200]
            if src == PRIMARY_QUOTE_SOURCE:
                kbs_fail_reason = last_reason
            logger.warning(
                "index %s (%s) exception: %s",
                code,
                src,
                str(e)[:120],
            )
            if src == PRIMARY_QUOTE_SOURCE:
                logger.warning(
                    "fallback KBS -> VCI cho index %s (exception từ KBS)",
                    code,
                )
            continue

    logger.warning(
        "Bỏ qua index %s — cả KBS và VCI đều không đạt (lý do cuối: %s)",
        code,
        last_reason,
    )
    return pd.DataFrame(), None


def build_index_target_schema(
    df: pd.DataFrame,
    index_code: str,
    run_date: str,
    *,
    src_used: str | None = None,
) -> pd.DataFrame:
    """Schema chỉ số: instrument_type=index + lineage."""
    return build_price_target_schema(
        df,
        index_code,
        run_date,
        src_used=src_used,
        instrument_type="index",
    )


In [5]:
def main():
    """Thu thập giá OHLCV."""
    _ensure_ingestion_workdir()
    run_dt = date.today()
    run_date = run_dt.isoformat()
    start_dt = _date_years_ago(run_dt, 3)
    start_s = start_dt.isoformat()
    end_s = run_dt.isoformat()

    logger.info("=" * 60)
    logger.info("PRICE INGESTION: %s ~ %s", start_s, end_s)
    logger.info("=" * 60)

    total = len(TICKERS)
    for idx, ticker in enumerate(TICKERS):
        logger.info("[%d/%d] Fetching %s...", idx + 1, total, ticker)
        try:
            raw, src_used = fetch_ticker_data(ticker, start_s, end_s)
            if raw.empty:
                logger.warning("Bỏ qua %s (rỗng)", ticker)
                continue
            cleaned = transform_data(raw)
            final_df = build_price_target_schema(
                cleaned, ticker, run_date, src_used=src_used, instrument_type="stock"
            )
            _log_ohlcv_quality(ticker, final_df, src_used)
            save_raw(final_df, "price", run_date, ticker)
            logger.info("✓ %s: %d rows", ticker, len(final_df))
        except Exception as e:
            logger.error("Lỗi %s: %s", ticker, e)

    logger.info("✓ Hoàn tất\n")

main()


2026-04-04 17:39:22 [INFO] ============================================================
2026-04-04 17:39:22 [INFO] PRICE INGESTION: 2023-04-04 ~ 2026-04-04
2026-04-04 17:39:22 [INFO] ============================================================
2026-04-04 17:39:22 [INFO] [1/20] Fetching ACB...
2026-04-04 17:39:24 [INFO] Lấy ACB từ KBS
2026-04-04 17:39:24 [INFO] ACB — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious', 'tradingDate']
2026-04-04 17:39:24 [INFO] ACB | src_used=KBS rows=748 min_date=2023-04-04 max_date=2026-04-03 pct_missing_close=0.00% pct_missing_value=0.00%
2026-04-04 17:39:24 [INFO] Đã ghi 748 dòng → c:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price\date=2026-04-04\ACB.parquet
2026-04-04 17:39:24 [INFO] ✓ ACB: 748 rows
2026-04-04 17:39:24 [INFO] [2/20] Fetching BCM...
2026-04-04 17:39:28 [INFO] Lấy BCM từ KB

In [6]:
# INDEX — chỉ số cơ sở (vnstock free: Quote + history 1D; xem .cursor/rules/instructions.md → Use Case 1 / 06-quote-price-api).
logger.info("⏸  Chờ %ss trước khi chuyển sang index...", DELAY_BETWEEN_CATEGORIES_SEC)
time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()
start_dt = _date_years_ago(run_dt, 3)
start_s = start_dt.isoformat()
end_s = run_dt.isoformat()

logger.info("=" * 60)
logger.info("INDEX INGESTION: %s ~ %s", start_s, end_s)
logger.info("=" * 60)

n_idx = len(INDEX_TICKERS)
for j, index_code in enumerate(INDEX_TICKERS):
    logger.info("[%d/%d] Fetching index %s...", j + 1, n_idx, index_code)
    try:
        cleaned_i, src_used_i = fetch_index_data(index_code, start_s, end_s)
        if cleaned_i.empty:
            logger.warning(
                "Bỏ qua index %s (rỗng / validation fail cả KBS+VCI)",
                index_code,
            )
            continue
        final_i = build_index_target_schema(
            cleaned_i, index_code, run_date, src_used=src_used_i
        )
        _log_ohlcv_quality(index_code, final_i, src_used_i)
        save_raw(final_i, "index", run_date, index_code)
        logger.info("✓ %s: %d rows", index_code, len(final_i))
    except Exception as e:
        logger.error("Lỗi index %s: %s", index_code, e)

logger.info("✓ Hoàn tất index\n")

2026-04-04 17:41:17 [INFO] ⏸  Chờ 60s trước khi chuyển sang index...
2026-04-04 17:42:17 [INFO] ============================================================
2026-04-04 17:42:17 [INFO] INDEX INGESTION: 2023-04-04 ~ 2026-04-04
2026-04-04 17:42:17 [INFO] ============================================================
2026-04-04 17:42:17 [INFO] [1/5] Fetching index VNINDEX...
2026-04-04 17:42:17 [INFO] Lấy VNINDEX từ KBS (validation OK)
2026-04-04 17:42:17 [INFO] VNINDEX — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious', 'tradingDate']
2026-04-04 17:42:17 [INFO] VNINDEX | src_used=KBS rows=748 min_date=2023-04-04 max_date=2026-04-03 pct_missing_close=0.00% pct_missing_value=0.00%
2026-04-04 17:42:17 [INFO] Đã ghi 748 dòng → c:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\index\date=2026-04-04\VNINDEX.parquet
2026-04-04 17:42:17 [IN

In [7]:
def fetch_listing() -> pd.DataFrame:
    """Master listing: all_symbols + exchange map từ symbols_by_exchange (2 nguồn KBS/VCI)."""

    def _standardize_columns(df_in: pd.DataFrame) -> pd.DataFrame:
        out = df_in.copy()
        out.columns = [str(c).strip().lower() for c in out.columns]
        return out

    def _jaccard(a: set[str], b: set[str]) -> float:
        if not a or not b:
            return 0.0
        inter = len(a & b)
        union = len(a | b)
        return float(inter / union) if union else 0.0

    def _build_exchange_map(source: str) -> tuple[pd.DataFrame, dict, bool]:
        """Trả về (emap[symbol,exchange], stats, unreliable). Chỉ dùng symbols_by_exchange."""
        exchanges = ["HOSE", "HNX", "UPCOM"]
        stats: dict = {
            "source": source,
            "counts": {e: 0 for e in exchanges},
            "jaccard": {},
            "n_fetched_exchanges": 0,
            "reasons": [],
        }
        empty = pd.DataFrame(columns=["symbol", "exchange"])

        try:
            listing = Listing(source=source)
        except Exception as e:
            stats["reasons"].append("listing_init_error:" + str(e)[:100])
            logger.warning("_build_exchange_map(%s): không khởi tạo Listing: %s", source, e)
            return empty, stats, True

        if not hasattr(listing, "symbols_by_exchange"):
            stats["reasons"].append("no_symbols_by_exchange")
            logger.warning(
                "Listing(source=%s) không có symbols_by_exchange — bỏ qua map",
                source,
            )
            return empty, stats, True

        sets_by_ex: dict[str, set[str]] = {}
        map_rows: list[dict[str, str]] = []

        for exc in exchanges:
            try:
                df_exc = listing.symbols_by_exchange(exchange=exc)
                if df_exc is None or df_exc.empty:
                    logger.info("symbols_by_exchange(%s) rỗng (source=%s)", exc, source)
                    continue
                df_exc = _standardize_columns(df_exc.copy())
                sym_col = None
                for cname in ("symbol", "ticker", "code"):
                    if cname in df_exc.columns:
                        sym_col = cname
                        break
                if not sym_col:
                    logger.warning(
                        "Không có cột symbol/ticker/code cho %s (source=%s)",
                        exc,
                        source,
                    )
                    continue
                syms = (
                    df_exc[sym_col]
                    .dropna()
                    .astype(str)
                    .str.strip()
                    .str.upper()
                )
                u = set(syms.unique())
                sets_by_ex[exc] = u
                stats["counts"][exc] = len(u)
                for s in u:
                    map_rows.append({"symbol": s, "exchange": exc})
            except Exception as e:
                logger.warning(
                    "symbols_by_exchange(%s) lỗi (source=%s): %s",
                    exc,
                    source,
                    e,
                )

        stats["n_fetched_exchanges"] = sum(
            1 for e in exchanges if sets_by_ex.get(e) and len(sets_by_ex[e]) > 0
        )

        pairs = [("HOSE", "HNX"), ("HOSE", "UPCOM"), ("HNX", "UPCOM")]
        high_j = False
        for a, b in pairs:
            sa, sb = sets_by_ex.get(a), sets_by_ex.get(b)
            if not sa or not sb:
                stats["jaccard"][f"{a}_{b}"] = None
                continue
            j = _jaccard(sa, sb)
            stats["jaccard"][f"{a}_{b}"] = round(j, 4)
            if j > 0.95:
                high_j = True

        unreliable = False
        n_ok = stats["n_fetched_exchanges"]

        if n_ok == 0:
            unreliable = True
            stats["reasons"].append("no_exchange_data")
        elif n_ok == 3:
            hose, hnx, up = (
                sets_by_ex.get("HOSE", set()),
                sets_by_ex.get("HNX", set()),
                sets_by_ex.get("UPCOM", set()),
            )
            if hose and hose == hnx == up:
                unreliable = True
                stats["reasons"].append("identical_hose_hnx_upcom")
        if high_j:
            unreliable = True
            stats["reasons"].append("jaccard_pair_gt_0.95")

        logger.info(
            "exchange map probe source=%s counts=%s n_fetched_exchanges=%s jaccard=%s unreliable=%s reasons=%s",
            source.upper(),
            stats["counts"],
            n_ok,
            stats["jaccard"],
            unreliable,
            stats["reasons"],
        )
        if unreliable:
            logger.warning(
                "exchange map UNRELIABLE source=%s — overlap/stale data: %s",
                source.upper(),
                "; ".join(stats["reasons"]) or "unknown",
            )

        if unreliable or not map_rows:
            return empty, stats, True

        emap = pd.DataFrame(map_rows)
        prio = {"HOSE": 0, "HNX": 1, "UPCOM": 2}
        emap["_p"] = emap["exchange"].map(prio)
        emap = emap.sort_values("_p").drop_duplicates("symbol", keep="first")
        emap = emap.drop(columns=["_p"]).reset_index(drop=True)
        return emap, stats, False

    logger.info(
        "⏸  Chờ %ss trước khi chuyển sang company listing...",
        DELAY_BETWEEN_CATEGORIES_SEC,
    )
    time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

    sources: tuple[str, ...] = (PRIMARY_QUOTE_SOURCE, FALLBACK_QUOTE_SOURCE)
    last_err: Exception | None = None
    df_base = pd.DataFrame()

    for src in sources:
        try:
            listing = Listing(source=src)
            df = listing.all_symbols()
            if df is None or (hasattr(df, "empty") and df.empty):
                logger.warning("all_symbols() trả rỗng với Listing source=%s", src)
                continue

            df_base = _standardize_columns(df)
            logger.info(
                "Lấy Listing thành công (%s mã) từ source=%s",
                len(df_base),
                src.upper(),
            )
            break

        except Exception as e:
            last_err = e
            logger.warning("Lỗi fetch_listing all_symbols source=%s: %s", src, e)
            continue

    if df_base.empty:
        logger.warning("Không lấy được Listing: %s", last_err)
        return pd.DataFrame()

    emap_final = pd.DataFrame(columns=["symbol", "exchange"])
    stats_used: dict = {}

    emap_p, stats_p, bad_p = _build_exchange_map(PRIMARY_QUOTE_SOURCE)
    if not bad_p:
        emap_final = emap_p
        stats_used = stats_p
        chosen_map_src = PRIMARY_QUOTE_SOURCE
        logger.info(
            "exchange map source used: %s (PRIMARY)",
            PRIMARY_QUOTE_SOURCE.upper(),
        )
    else:
        emap_f, stats_f, bad_f = _build_exchange_map(FALLBACK_QUOTE_SOURCE)
        if not bad_f:
            emap_final = emap_f
            stats_used = stats_f
            chosen_map_src = FALLBACK_QUOTE_SOURCE
            logger.info(
                "exchange map source used: %s (fallback sau PRIMARY unreliable)",
                FALLBACK_QUOTE_SOURCE.upper(),
            )
        else:
            logger.warning(
                "exchange map unavailable (PRIMARY + FALLBACK đều unreliable) "
                "→ merge không có map, exchange=NA, exchange_source_unreliable=True"
            )
            stats_used = {"primary_stats": stats_p, "fallback_stats": stats_f}

    df_result = df_base.copy()
    if "symbol" not in df_result.columns:
        logger.warning("listing_all không có cột symbol")
        return df_result

    df_result["symbol"] = (
        df_result["symbol"].astype(str).str.strip().str.upper()
    )

    df_result = df_result.drop(columns=["exchange"], errors="ignore")

    if emap_final.empty:
        df_result["exchange"] = pd.NA
        df_result["exchange_source_unreliable"] = True
    else:
        df_result = df_result.merge(emap_final, on="symbol", how="left")
        coverage_partial = int(stats_used.get("n_fetched_exchanges", 0)) < 3
        df_result["exchange_source_unreliable"] = bool(coverage_partial)
        if coverage_partial:
            logger.warning(
                "exchange map partial coverage (n_fetched_exchanges=%s < 3) "
                "→ exchange_source_unreliable=True (vẫn join phần map có được)",
                stats_used.get("n_fetched_exchanges"),
            )

    before_filter = len(df_result)

    if "type" in df_result.columns:
        df_stock = df_result[df_result["type"] == "stock"].copy()
    elif "id" in df_result.columns:
        df_stock = df_result[df_result["id"] == 1].copy()
    else:
        df_stock = df_result.copy()

    after_filter = len(df_stock)
    logger.info(
        "Lọc stock: %s → %s dòng (bỏ %s non-stock)",
        before_filter,
        after_filter,
        before_filter - after_filter,
    )
    df_result = df_stock

    if "organ_name_x" in df_result.columns:
        df_result = df_result.rename(columns={"organ_name_x": "organ_name"})
    if "organ_name_y" in df_result.columns:
        df_result = df_result.drop(columns=["organ_name_y"])
    if "id" in df_result.columns:
        df_result = df_result.drop(columns=["id"])

    df_result["crawled_at"] = date.today().strftime("%Y-%m-%d")

    priority_cols = [
        "symbol",
        "organ_name",
        "en_organ_name",
        "exchange",
        "exchange_source_unreliable",
        "type",
        "crawled_at",
    ]
    existing = [c for c in priority_cols if c in df_result.columns]
    other = [c for c in df_result.columns if c not in existing]
    df_result = df_result[existing + other]

    logger.info("Listing columns cuối: %s", df_result.columns.tolist())
    if "exchange" in df_result.columns:
        logger.info(
            "Phân bổ exchange cuối (value_counts):\n%s",
            df_result["exchange"].value_counts(dropna=False).to_string(),
        )

    return df_result


# --- Run Listing ---
logger.info("=" * 70)
logger.info("LISTING (Master data)")
logger.info("=" * 70)

_ensure_ingestion_workdir()
df_listing = fetch_listing()

if df_listing.empty:
    logger.warning("Bỏ qua lưu Listing vì DataFrame rỗng.")
else:
    print(df_listing[["symbol", "exchange", "exchange_source_unreliable"]].head(20))
    print(df_listing["exchange"].value_counts(dropna=False))

    try:
        base_dir = os.path.abspath(
            os.path.join(os.getcwd(), "..", "data-lake", "raw", "Structure_Data", "listing", "master")
        )
        os.makedirs(base_dir, exist_ok=True)

        parquet_path = os.path.join(base_dir, "listing.parquet")
        csv_path = os.path.join(base_dir, "listing.csv")

        try:
            df_listing.to_parquet(parquet_path, engine="pyarrow", index=False)
        except ImportError:
            df_listing.to_parquet(parquet_path, engine="fastparquet", index=False)
        df_listing.to_csv(csv_path, index=False, encoding="utf-8-sig")

        logger.info("✓ Đã ghi %s dòng → %s", len(df_listing), parquet_path)
        logger.info("✓ Đã xuất %s dòng → %s", len(df_listing), csv_path)
    except Exception:
        logger.exception("✗ Lỗi khi lưu Listing")


2026-04-04 17:42:41 [INFO] ======================================================================
2026-04-04 17:42:41 [INFO] LISTING (Master data)
2026-04-04 17:42:41 [INFO] ======================================================================
2026-04-04 17:42:41 [INFO] ⏸  Chờ 60s trước khi chuyển sang company listing...
2026-04-04 17:43:42 [INFO] Lấy Listing thành công (1544 mã) từ source=KBS
2026-04-04 17:43:43 [INFO] exchange map probe source=KBS counts={'HOSE': 1942, 'HNX': 1942, 'UPCOM': 1942} n_fetched_exchanges=3 jaccard={'HOSE_HNX': 1.0, 'HOSE_UPCOM': 1.0, 'HNX_UPCOM': 1.0} unreliable=True reasons=['identical_hose_hnx_upcom', 'jaccard_pair_gt_0.95']
2026-04-04 17:43:43 [WARNING] exchange map UNRELIABLE source=KBS — overlap/stale data: identical_hose_hnx_upcom; jaccard_pair_gt_0.95
2026-04-04 17:43:45 [INFO] exchange map probe source=VCI counts={'HOSE': 3314, 'HNX': 3314, 'UPCOM': 3314} n_fetched_exchanges=3 jaccard={'HOSE_HNX': 1.0, 'HOSE_UPCOM': 1.0, 'HNX_UPCOM': 1.0} unrelia

   symbol exchange  exchange_source_unreliable
0     MGC     <NA>                        True
1     GVT     <NA>                        True
2     GEG     <NA>                        True
3     SWC     <NA>                        True
4     SLD     <NA>                        True
5     VID     <NA>                        True
6     PSL     <NA>                        True
7     TTZ     <NA>                        True
8     CX8     <NA>                        True
9     PNJ     <NA>                        True
10    TMB     <NA>                        True
11    NTH     <NA>                        True
12    CVT     <NA>                        True
13    POV     <NA>                        True
14    SAM     <NA>                        True
15    SDV     <NA>                        True
16    MTH     <NA>                        True
17    KHG     <NA>                        True
18    NWT     <NA>                        True
19    BAX     <NA>                        True
exchange
<NA>

In [11]:
def fetch_company_overview(ticker: str) -> dict:
    """Lấy thông tin tổng quan doanh nghiệp cho một mã.

    Không raise exception; lỗi sẽ trả về dict {'ticker': ticker}.
    """
    sym = ticker.upper().strip()
    try:
        company = Company(source=PRIMARY_QUOTE_SOURCE, symbol=sym)

        result = None
        try:
            if hasattr(company, "overview"):
                result = company.overview()
            elif hasattr(company, "profile"):
                result = company.profile()
            else:
                logger.warning("Company: không có method overview/profile (ticker=%s)", sym)
                result = {}
        except Exception as e:
            logger.warning("Company method lỗi %s: %s", sym, e)
            result = {}

        # Chuẩn hóa về dict
        if result is None:
            out: dict = {}
        elif isinstance(result, dict):
            out = dict(result)
        elif isinstance(result, pd.Series):
            out = result.to_dict()
        elif isinstance(result, pd.DataFrame):
            out = result.iloc[0].to_dict() if not result.empty else {}
        else:
            try:
                out = dict(result)
            except Exception:
                out = {}

        out["ticker"] = sym
        return out

    except Exception as e:
        logger.warning("fetch_company_overview lỗi %s: %s", sym, e)
        return {"ticker": sym}


# --- Run company overview ---
logger.info("⏸  Chờ %ss trước khi chuyển sang company overview...", DELAY_BETWEEN_CATEGORIES_SEC)
time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()

logger.info("=" * 70)
logger.info("COMPANY OVERVIEW | %s mã", len(TICKERS))
logger.info("=" * 70)

rows: list[dict] = []
total = len(TICKERS)
for idx, ticker in enumerate(TICKERS):
    logger.info("[%s/%s] Đang tải company %s...", idx + 1, total, ticker)
    try:
        rows.append(fetch_company_overview(ticker))
        logger.info("✓ OK company %s", ticker)
    except Exception:
        logger.exception("✗ Lỗi khi xử lý company %s — tiếp tục", ticker)

    if idx < total - 1:
        time.sleep(0.5)

company_df = pd.DataFrame(rows)

try:
    if company_df.empty:
        logger.warning("company_df rỗng — bỏ qua lưu.")
    else:
        # Xử lý các cột text dài: replace newline bằng space
        text_cols = company_df.select_dtypes(include="object").columns
        for col in text_cols:
            company_df[col] = (
                company_df[col]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.replace("\r", " ", regex=False)
                .str.strip()
            )
        
        save_raw(company_df, "company", run_date, "company_overview")
        logger.info("✓ Đã lưu company overview\n")
except Exception:
    logger.exception("✗ Lỗi khi lưu company_overview")

2026-04-04 17:49:01 [INFO] ⏸  Chờ 60s trước khi chuyển sang company overview...
2026-04-04 17:50:01 [INFO] ======================================================================
2026-04-04 17:50:01 [INFO] COMPANY OVERVIEW | 20 mã
2026-04-04 17:50:01 [INFO] ======================================================================
2026-04-04 17:50:01 [INFO] [1/20] Đang tải company ACB...
2026-04-04 17:50:01 [INFO] ✓ OK company ACB
2026-04-04 17:50:02 [INFO] [2/20] Đang tải company BCM...
2026-04-04 17:50:02 [INFO] ✓ OK company BCM
2026-04-04 17:50:02 [INFO] [3/20] Đang tải company BID...
2026-04-04 17:50:03 [INFO] ✓ OK company BID
2026-04-04 17:50:03 [INFO] [4/20] Đang tải company BVH...
2026-04-04 17:50:03 [INFO] ✓ OK company BVH
2026-04-04 17:50:04 [INFO] [5/20] Đang tải company CTG...
2026-04-04 17:50:04 [INFO] ✓ OK company CTG
2026-04-04 17:50:05 [INFO] [6/20] Đang tải company FPT...
2026-04-04 17:50:05 [INFO] ✓ OK company FPT
2026-04-04 17:50:06 [INFO] [7/20] Đang tải company GAS...
20

In [8]:
def fetch_financial_ratio(ticker: str) -> pd.DataFrame:
    """Lấy financial ratio theo quý (fallback year nếu lỗi)."""
    sym = ticker.upper().strip()
    try:
        finance = Finance(source=PRIMARY_QUOTE_SOURCE, symbol=sym)

        df = pd.DataFrame()
        try:
            df = finance.ratio(period="quarter")
        except Exception:
            df = finance.ratio(period="year")

        if df is None or (hasattr(df, "empty") and df.empty):
            return pd.DataFrame()

        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df)

        out = df.copy()
        out["ticker"] = sym
        return out

    except Exception as e:
        logger.warning("fetch_financial_ratio lỗi %s: %s", sym, e)
        return pd.DataFrame()


# --- Run financial_ratio ---
logger.info("⏸  Chờ %ss trước khi chuyển sang financial_ratio...", DELAY_BETWEEN_CATEGORIES_SEC)
time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()

logger.info("=" * 70)
logger.info("FINANCIAL RATIO | %s mã (theo quarter)", len(TICKERS))
logger.info("=" * 70)

total = len(TICKERS)
for idx, ticker in enumerate(TICKERS):
    logger.info("[%s/%s] Đang tải financial ratio %s...", idx + 1, total, ticker)
    try:
        df_ratio = fetch_financial_ratio(ticker)
        if df_ratio.empty:
            logger.warning("Bỏ qua %s — financial_ratio rỗng", ticker)
        else:
            save_raw(df_ratio, "financial_ratio", run_date, ticker)
            logger.info("✓ Thành công ratio %s (%s dòng)", ticker, len(df_ratio))
    except Exception:
        logger.exception("✗ Lỗi khi xử lý financial_ratio %s — tiếp tục", ticker)

    if idx < total - 1:
        time.sleep(0.5)  # Giảm từ 1s xuống 0.5s

logger.info("✓ Hoàn tất financial_ratio cho %s mã.\n", total)

2026-04-04 17:43:45 [INFO] ⏸  Chờ 60s trước khi chuyển sang financial_ratio...
2026-04-04 17:44:45 [INFO] ======================================================================
2026-04-04 17:44:45 [INFO] FINANCIAL RATIO | 20 mã (theo quarter)
2026-04-04 17:44:45 [INFO] ======================================================================
2026-04-04 17:44:45 [INFO] [1/20] Đang tải financial ratio ACB...
2026-04-04 17:44:45 [INFO] Loaded 162 built-in KBS field mappings
2026-04-04 17:44:45 [INFO] Loaded 162 built-in KBS field mappings
2026-04-04 17:44:46 [INFO] Đã ghi 32 dòng → c:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\financial_ratio\date=2026-04-04\ACB.parquet
2026-04-04 17:44:46 [INFO] ✓ Thành công ratio ACB (32 dòng)
2026-04-04 17:44:46 [INFO] [2/20] Đang tải financial ratio BCM...
2026-04-04 17:44:46 [INFO] Loaded 162 built-in KBS field mappings
2026-04-04 17:44:46 [INFO] Loaded 162 built-in KBS field mappings
2026-04-04 17:44:47 [INFO] Đã ghi 32 dò

In [9]:
def fetch_price_board(tickers_list: list[str]) -> pd.DataFrame:
    """Thu thập snapshot bảng giá tham chiếu/trần/sàn.

    Nếu API/method không khả dụng sẽ log warning và trả DataFrame rỗng.
    """
    from datetime import datetime, time as dtime
    import re

    required_norm_map = {
        "ticker": "ticker",
        "ref_price": "ref_price",
        "ceiling_price": "ceiling_price",
        "floor_price": "floor_price",
        "match_price": "match_price",
        "match_volume": "match_volume",
        "foreign_buy_vol": "foreign_buy_vol",
        "foreign_sell_vol": "foreign_sell_vol",
        "bid_price1": "bid_price1",
        "bid_vol1": "bid_vol1",
        "ask_price1": "ask_price1",
        "ask_vol1": "ask_vol1",
    }

    def _norm_col(s: str) -> str:
        return re.sub(r"[\s_]+", "", str(s).strip().lower())

    # Chỉ chạy trong giờ giao dịch theo yêu cầu.
    now_t = datetime.now().time()
    if not (dtime(9, 0) <= now_t <= dtime(15, 0)):
        logger.info("Ngoài giờ giao dịch (now=%s) — bỏ qua price_board.", now_t.strftime("%H:%M:%S"))
        return pd.DataFrame()

    if not tickers_list:
        return pd.DataFrame()

    try:
        probe = Quote(source=PRIMARY_QUOTE_SOURCE, symbol=tickers_list[0])
        if not hasattr(probe, "price_board"):
            logger.warning("Method `Quote.price_board()` không khả dụng trong vnstock version này.")
            return pd.DataFrame()

        df_out = pd.DataFrame()

        # 1) Thử lấy nhiều mã cùng lúc (nếu method hỗ trợ tham số symbols/tickers).
        try:
            try:
                df_out = probe.price_board(symbols=tickers_list)
            except TypeError:
                df_out = probe.price_board(tickers_list)
        except Exception:
            df_out = pd.DataFrame()

        # 2) Fallback: lấy lần lượt từng mã nếu không có multi.
        if df_out is None or (hasattr(df_out, "empty") and df_out.empty):
            frames: list[pd.DataFrame] = []
            for t in tickers_list:
                try:
                    q = Quote(source=PRIMARY_QUOTE_SOURCE, symbol=t)
                    sub = q.price_board()
                    if sub is None or (hasattr(sub, "empty") and sub.empty):
                        continue
                    if not isinstance(sub, pd.DataFrame):
                        sub = pd.DataFrame(sub)
                    sub = sub.copy()
                    if "ticker" not in sub.columns:
                        sub["ticker"] = t
                    frames.append(sub)
                except Exception as e:
                    logger.warning("price_board lỗi %s: %s", t, e)

            if frames:
                df_out = pd.concat(frames, ignore_index=True)

        if df_out is None or (hasattr(df_out, "empty") and df_out.empty):
            return pd.DataFrame()

        if not isinstance(df_out, pd.DataFrame):
            df_out = pd.DataFrame(df_out)

        # Chỉ giữ các cột cần nếu có.
        col_map = {_norm_col(c): c for c in df_out.columns}
        required_norms = [_norm_col(k) for k in required_norm_map.keys()]
        keep_cols = [col_map[rn] for rn in required_norms if rn in col_map]
        out = df_out.loc[:, keep_cols].copy() if keep_cols else df_out.copy()

        # Đảm bảo cột ticker luôn có.
        if "ticker" not in [_norm_col(c) for c in out.columns] and "ticker" not in out.columns:
            out["ticker"] = tickers_list[0]

        return out

    except Exception as e:
        logger.warning("fetch_price_board lỗi: %s", e)
        return pd.DataFrame()


# --- Run price_board snapshot ---
logger.info("⏸  Chờ %ss trước khi chuyển sang price_board...", DELAY_BETWEEN_CATEGORIES_SEC)
time.sleep(DELAY_BETWEEN_CATEGORIES_SEC)

_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()

logger.info("=" * 70)
logger.info("PRICE BOARD SNAPSHOT | %s mã", len(TICKERS))
logger.info("=" * 70)

try:
    board_df = fetch_price_board(TICKERS)
    if board_df.empty:
        logger.warning("price_board_snapshot rỗng — bỏ qua lưu.")
    else:
        save_raw(board_df, "price_board", run_date, "price_board_snapshot")
        logger.info("✓ Đã lưu price_board snapshot\n")
except Exception:
    logger.exception("✗ Lỗi khi thu thập/lưu price_board")

logger.info("=" * 70)
logger.info("✓ INGESTION HOÀN TẤT")
logger.info("=" * 70)

2026-04-04 17:45:08 [INFO] ⏸  Chờ 60s trước khi chuyển sang price_board...
2026-04-04 17:46:08 [INFO] ======================================================================
2026-04-04 17:46:08 [INFO] PRICE BOARD SNAPSHOT | 20 mã
2026-04-04 17:46:08 [INFO] ======================================================================
2026-04-04 17:46:08 [INFO] Ngoài giờ giao dịch (now=17:46:08) — bỏ qua price_board.
2026-04-04 17:46:08 [WARNING] price_board_snapshot rỗng — bỏ qua lưu.
2026-04-04 17:46:08 [INFO] ======================================================================
2026-04-04 17:46:08 [INFO] ✓ INGESTION HOÀN TẤT
2026-04-04 17:46:08 [INFO] ======================================================================


In [12]:
# CELL 9 — Kiểm tra tổng kết dữ liệu theo từng category.

import logging
from datetime import date
from pathlib import Path

import pandas as pd

logger = globals().get("logger", logging.getLogger(__name__))


def _datalake_raw_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "ingest_structure_data.ipynb").is_file() or (
        cwd / "ingest_price_data.ipynb"
    ).is_file():
        base = cwd
    elif (cwd / "ingestion" / "ingest_structure_data.ipynb").is_file() or (
        cwd / "ingestion" / "ingest_price_data.ipynb"
    ).is_file():
        base = cwd / "ingestion"
    else:
        base = cwd
    return (base / ".." / "data-lake" / "raw" / "Structure_Data").resolve()


def _get_null_columns(df: pd.DataFrame, threshold: float = 50.0) -> list[str]:
    """Trả về danh sách cột có tỷ lệ null > threshold%."""
    null_pct = (df.isnull().sum() / len(df) * 100)
    return [col for col, pct in null_pct.items() if pct > threshold]


raw_root = _datalake_raw_root()
run_date = date.today().isoformat()

categories = {
    "price":           f"../data-lake/raw/price/date={run_date}",
    "index":           f"../data-lake/raw/index/date={run_date}",
    "listing":         f"../data-lake/raw/listing/master",
    "company":         f"../data-lake/raw/company/date={run_date}",
    "financial_ratio": f"../data-lake/raw/financial_ratio/date={run_date}",
    "price_board":     f"../data-lake/raw/price_board/date={run_date}",
}


def _read_parquet_rows(p: Path) -> int:
    try:
        df = pd.read_parquet(p)
        return int(len(df))
    except Exception as e:
        logger.warning("QC: đọc lỗi %s: %s", p, e)
        return 0


def _get_null_cols_for_file(p: Path) -> str:
    """Đọc file parquet và trả về danh sách cột nullvalue > 50%."""
    try:
        df = pd.read_parquet(p)
        null_cols = _get_null_columns(df, threshold=50.0)
        if null_cols:
            return ", ".join(null_cols)
        else:
            return ""
    except Exception as e:
        logger.warning("QC: đọc null_cols lỗi %s: %s", p, e)
        return ""


summary_rows: list[dict] = []
missing_or_empty: list[str] = []

for cat, path_hint in categories.items():
    total_rows = 0
    file_count = 0
    null_cols_list: list[str] = []

    if cat == "listing":
        p = raw_root / "listing" / "master" / "listing.parquet"
        if p.is_file():
            file_count = 1
            total_rows = _read_parquet_rows(p)
            null_cols_list.append(_get_null_cols_for_file(p))
    else:
        folder = raw_root / cat / f"date={run_date}"
        if folder.is_dir():
            parquets = sorted(folder.glob("*.parquet"))
            file_count = len(parquets)
            total_rows = sum(_read_parquet_rows(x) for x in parquets)
            # Lấy null_cols từ file đầu tiên để represent category
            if parquets:
                null_cols_list.append(_get_null_cols_for_file(parquets[0]))

    if file_count == 0:
        status = "MISSING"
    elif total_rows == 0:
        status = "EMPTY"
    else:
        status = "OK"

    if status != "OK":
        missing_or_empty.append(cat)

    null_cols_str = "; ".join([s for s in null_cols_list if s])

    summary_rows.append(
        {
            "Category": cat,
            "Files": file_count,
            "Total rows": total_rows,
            "Status": status,
            "Null cols (>50%)": null_cols_str if null_cols_str else "-",
        }
    )


print("Category           | Files | Total rows | Status | Null cols (>50%)")
print("------------------|-------|------------|--------|------------------")
for r in summary_rows:
    null_str = r["Null cols (>50%)"]
    print(
        f"{r['Category']:<18} | {r['Files']:<5} | {r['Total rows']:<10,} | {r['Status']:<6} | {null_str}"
    )

if missing_or_empty:
    print("\n[QC WARNING] Các category bị thiếu hoặc rỗng:")
    for cat in missing_or_empty:
        print(f"- {cat}")
else:
    print("\n[QC] Tất cả category có dữ liệu.")

Category           | Files | Total rows | Status | Null cols (>50%)
------------------|-------|------------|--------|------------------
price              | 20    | 14,961     | OK     | -
index              | 5     | 3,739      | OK     | -
listing            | 1     | 1,544      | OK     | exchange
company            | 1     | 20         | OK     | -
financial_ratio    | 20    | 631        | OK     | -
price_board        | 0     | 0          | MISSING | -

[QC WARNING] Các category bị thiếu hoặc rỗng:
- price_board
